### Agent + Short-Term Memory

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    system_prompt="You are a helpful assistant.",
    checkpointer=InMemorySaver()
)

config = {
    "configurable": {
        "thread_id": "user-123"
    }
}

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Alok and I am learning GenAI."
            }
        ]
    },
    config
)

print(response["messages"][-1].text)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What am I learning?"
            }
        ]
    },
    config
)

print(response["messages"][-1].text)

### InMemorySaver vs Database

In [ ]:
# Development
# InMemorySaver()




# Good for:

# Jupyter
# experiments
# learning
# prototypes




# But if the Python process stops:

# memory disappears





# Production

# Use a persistent checkpointer.

# For example PostgreSQL.



from langgraph.checkpoint.postgres import PostgresSaver

# with a PostgreSQL connection.

# Conceptually:

# Agent
#  ↓
# PostgreSQL checkpointer
#  ↓
# thread state
#  ↓
# survives restart

### Short-Term Memory ≠ Long-Term Memory

In [ ]:
# This distinction is extremely important.



# Short-term
# thread_id = conversation-123

# messages:
#     user
#     AI
#     user
#     AI




# Long-term
# user_id = 123

# preferences:
#     Python
#     concise explanations

# facts:
#     student
#     building AI application




# Think:

# THREAD MEMORY
#       +
# USER MEMORY




# not:

# one giant memory

### Custom Agent State

In [ ]:
# Modern LangChain lets you extend the agent state.

# Example:

from langchain.agents import AgentState, create_agent
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):
    user_id: str
    preferences: dict

# Then:

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[],
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver()
)

# Invoke:

config = {
    "configurable": {
        "thread_id": "thread-1"
    }
}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hello!"
            }
        ],
        "user_id": "user-123",
        "preferences": {
            "language": "Python"
        }
    },
    config
)

print(result.content)

# Current LangChain documents AgentState and custom state schemas as the mechanism 
# for adding fields beyond the default message history.

### Memory Inside Tools

In [ ]:
# This becomes very powerful with agents.

# A tool can access the agent's current state.



# Conceptually:

# User
#  ↓
# Agent
#  ↓
# Tool
#  ↓
# Tool reads memory
#  ↓
# Result
#  ↓
# Agent

# Modern LangChain provides ToolRuntime for this.




# Example:

# from langchain.tools import tool, ToolRuntime

# @tool
# def get_user_name(runtime: ToolRuntime) -> str:
#     """Get the user's name from agent state."""
    
#     user_id = runtime.state.get("user_id")

#     if user_id == "user-123":
#         return "Alok"

#     return "Unknown user"




# Then:

# agent = create_agent(
#     model="google_genai:gemini-3.6-flash",
#     tools=[get_user_name],
#     state_schema=CustomAgentState
# )

# The current LangChain memory documentation explicitly
#  demonstrates accessing short-term state from tools using ToolRuntime.

### Long-Term Memory Architecture

In [ ]:
# For a serious application:

#                     USER
#                       │
#                       ↓
#                     AGENT
#                       │
#             ┌─────────┴─────────┐
#             ↓                   ↓
#      Short-term memory     Long-term memory
#             │                   │
#        Checkpointer          Store
#             │                   │
#        thread state       user information
#             │                   │
#             └─────────┬─────────┘
#                       ↓
#                     GEMINI

### Long-Term Memory Store

In [ ]:
# The current LangChain architecture uses a persistent Store for long-term information, separate from the checkpointer used for thread state.

# Conceptually:

# Checkpointer
#     ↓
# "What happened in this conversation?"

# Store
#     ↓
# "What should I remember about this user/application?"

# This distinction is much more important for you than memorizing the old VectorStoreRetrieverMemory class.

### Memory + RAG

In [ ]:
# You already learned RAG.

# Normal RAG:

# Question
#  ↓
# Retriever
#  ↓
# Documents
#  ↓
# Gemini



# Memory-enhanced RAG:

# Question
#  ↓
#  ┌───────────────┐
#  ↓               ↓
# RAG Retriever   Memory Retriever
#  ↓               ↓
# Documents       User context
#  └───────┬───────┘
#          ↓
#        Prompt
#          ↓
#        Gemini




# Example:

# User:
# "Explain this PDF using the level I prefer."

# RAG:
# → retrieves PDF information

# Memory:
# → retrieves "user prefers beginner explanations"

# Gemini:
# → combines both



# This is extremely useful in your StudyFlow-style application.

### Memory + Agents

In [ ]:
# Now combine everything:

#                     User
#                      ↓
#                   Agent
#                      │
#        ┌─────────────┼─────────────┐
#        ↓             ↓             ↓
#  Short-term      Long-term       Tools
#   memory          memory
#        │             │             │
#        └─────────────┼─────────────┘
#                      ↓
#                    Gemini

# The agent can decide:

# "Do I need a tool?"
# "Do I need retrieved memory?"
# "What part of conversation is relevant?"

### Multi-User Memory

In [ ]:
# Real application:

# user_id
#    ↓
# thread_id

# For example:

# user-001
#  ├── thread-001
#  ├── thread-002
#  └── thread-003

# user-002
#  ├── thread-001
#  └── thread-002

# Important:

# user_id ≠ thread_id
# user_id

# Identifies the person/application owner.

# thread_id

# Identifies one conversation/session.

# This distinction is essential for production chat applications.

### Memory Persistence

In [ ]:
# You should understand three levels:

# Level 1 — Notebook memory
# InMemorySaver()
# temporary


# Level 2 — Persistent conversation state
# PostgreSQL
# SQLite
# other checkpointer backend
# conversation survives restart



# Level 3 — Long-term user memory
# Store
# Vector DB
# Database
# information survives across conversations

### Memory and Privacy

In [ ]:
# This is often overlooked.



# If an application stores:

# name
# email
# preferences
# conversation
# documents



# you need policies for:

# What do we store?
# Why do we store it?
# How long?
# Who can access it?
# Can the user delete it?
# Can the user update it?



# Never blindly store every conversation forever.

### Memory Retrieval Should Be Selective

In [ ]:
# Don't do:

# retrieve every memory



# Do:

# current query
#       ↓
# retrieve relevant memories
#       ↓
# top-k
#       ↓
# Gemini




# For example:

# User has 10,000 memories.



# Current question:
# "What programming language should I use?"



# Retrieve:
# → programming preference
# → project context



# Ignore:
# → old restaurant discussion
# → unrelated conversations


# That's the real purpose of semantic memory.

### Memory + Vector Store vs RAG

In [ ]:
# These look similar:



# RAG
# documents
#  ↓
# embeddings
#  ↓
# vector DB
#  ↓
# retrieval




# Memory
# user facts/conversations
#  ↓
# embeddings
#  ↓
# vector DB
#  ↓
# retrieval


# Technically they use very similar infrastructure.

# The difference is what you're retrieving.

# RAG → external knowledge
# Memory → previous user/application information

### Complete Modern Memory Example

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver


checkpointer = InMemorySaver()

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[],
    system_prompt=(
        "You are a helpful assistant. "
        "Use conversation context when relevant."
    ),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.6-flash",
            trigger=("tokens", 4000),
            keep=("messages", 20)
        )
    ],
    checkpointer=checkpointer
)


config = {
    "configurable": {
        "thread_id": "user-123"
    }
}


response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Alok."
            }
        ]
    },
    config
)

print(response["messages"][-1].text)


response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my name?"
            }
        ]
    },
    config
)

print(response["messages"][-1].text)

In [ ]:
# The architecture is:

#                     AGENT
#                       │
#                AgentState
#                       │
#               ┌───────┴───────┐
#               │               │
#          Messages         Custom state
#               │
#         Checkpointer
#               │
#          thread_id
#               │
#        Context management
#         /       |        \
#      trim    delete    summarize

### Final Memory Architecture You Should Remember

In [ ]:
# For your AI-engineering learning path, memorize this:

#                          MEMORY
#                             │
#              ┌──────────────┴──────────────┐
#              │                             │
#        SHORT-TERM                      LONG-TERM
#              │                             │
#         one thread                    across threads
#              │                             │
#        AgentState                         Store
#              │                             │
#        Checkpointer                 DB / Vector Store
#              │                             │
#        thread_id                    user_id / namespace
#              │                             │
#       ┌──────┼──────┐                    │
#       │      │      │                    │
#     full   trim  summary             semantic search
#    history  │      │                    │
#             └──────┘                    │
#                  │                      │
#                  └──────────┬───────────┘
#                             ↓
#                           AGENT
#                             ↓
#                           GEMINI



# PART 1
# Memory fundamentals
#         ↓
# Messages + manual history
#         ↓
# Modern AgentState
#         ↓
# Checkpointer + thread_id




# PART 2
# Buffer concept
#         ↓
# Window concept
#         ↓
# Trim messages
#         ↓
# Delete messages
#         ↓
# Token management





# PART 3
# Summary concept
#         ↓
# Running summaries
#         ↓
# Hybrid memory
#         ↓
# SummarizationMiddleware




# PART 4
# Embeddings
#         ↓
# Vector store
#         ↓
# Semantic memory
#         ↓
# Long-term user memory
#         ↓
# Memory retrieval





# PART 5
# Agent + memory
#         ↓
# Persistent checkpointer
#         ↓
# Long-term Store
#         ↓
# Custom state
#         ↓
# Memory inside tools
#         ↓
# Multi-user memory
#         ↓
# Memory + RAG
#         ↓
# Memory + agents
#         ↓
# CRUD / privacy / security
#         ↓
# Production architecture